# SURENA VLA Episode Evaluation — single & batch (×10) + video

Evaluates `EpisodeLogger` `.h5` episode files: **single** or **batch** (10 runs per preset).

* **Input** (`H5_PATH` single, or `H5_INPUT` = dir / glob / list for batch) — resolver auto-detects.
* **Reporting rule**: batch tables use **mean ± std**, batch figures use **mean ± 95% CI** (t, df=n−1).
* **Video**: rebuilt from the file's own JPEG streams (no re-simulation) — plain MP4 + animated frame|action-history.

**Sections** 1 imports/style/input · 2 HDF5 inspect · 3 signals · 4 extraction · 5 time base · 6 summary · 7 EEF/joints · 8 video · 9 motion · 10 smoothness · 11 effort · 12 IK (pie) · 13 VLA actions (split + combined 7) · 14 timing · 15 success · 16 final tables + export


## 1. Imports, palette, style, input resolver

In [ ]:
import os, re, glob, json, math, warnings
from pathlib import Path
from collections import defaultdict, Counter
from matplotlib.ticker import MaxNLocator

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy import stats as scipy_stats
from IPython.display import HTML, display

# paper style: serif/Times, white face, light grid (user gist)
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 320, "font.size": 12,
    "font.family": "serif",
    "font.serif": ["Times New Roman"] + plt.rcParams["font.serif"],
    "axes.facecolor": "white", "axes.grid": True, "grid.color": ".8",
    "axes.edgecolor": "0.15", "axes.linewidth": 0.8,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "legend.frameon": True, "legend.framealpha": 0.9, "legend.edgecolor": "#cccccc",
})

# Tol muted / Okabe-Ito blend - colorblind-safe, print-friendly.
VLA_COLORS_7 = ["#4477AA", "#EE6677", "#228833", "#CCBB44", "#66CCEE", "#AA3377", "#505050"]
VLA_LABELS_7 = ["dx", "dy", "dz", "droll", "dpitch", "dyaw", "gripper"]
XYZ_COLORS    = ["#4477AA", "#EE6677", "#228833"]
RPY_COLORS    = ["#CCBB44", "#66CCEE", "#AA3377"]
GRIP_COLOR    = "#505050"
IK_PIE_COLORS = {"full": "#228833", "pos": "#F59F00", "fail": "#C92A2A"}
CI_ALPHA = 0.3          # single 95% CI band, like the gist (no std band)

FIG_SQUARE   = (7.0, 5.2)   # square-ish main figures
FIG_SQ_SMALL = (6.2, 4.6)   # pie / small
FIG_WIDE_2   = (10.5, 5.0)  # 3d+2d eef
FIG_COMB     = (7.5, 5.4)   # combined 7-dim
FIG_HIST     = (6.6, 4.0)

def _style_ax(ax, *, xlabel=None, ylabel=None, title=None, legend=True, yticks5=True):
    ax.grid(True, color=".8", linewidth=0.7)
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)
    if title:  ax.set_title(title, fontsize=9) 
    ax.tick_params(labelsize=7)
    if yticks5:
        try: ax.yaxis.set_major_locator(MaxNLocator(5))
        except Exception: pass
    if legend:
        h, l = ax.get_legend_handles_labels()
        if h: ax.legend(fontsize=8, loc="best")
    for sp in ax.spines.values(): sp.set_color("0.15")

def ci95_factor(n):
    """t(0.975, df=n-1) for 95% CI; NaN when undefined."""
    n = int(n)
    return float(scipy_stats.t.ppf(0.975, df=n - 1)) if n >= 2 else float("nan")

def mean_std_ci(vals, axis=0):
    """Returns (mean, std ddof=1, ci95) collapsing `axis`; NaN-aware."""
    a = np.asarray(vals, dtype=float)
    n = a.shape[axis] if a.ndim else 1
    mean = np.nanmean(a, axis=axis) if n else np.asarray([])
    std = np.nanstd(a, axis=axis, ddof=1) if n > 1 else np.full(np.shape(mean), np.nan)
    sem = std / np.sqrt(max(n, 1))
    t95 = ci95_factor(n)
    ci = t95 * sem if np.isfinite(t95) else np.full(np.shape(mean), np.nan)
    return mean, std, ci, n

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")
print("Imports + style ready (serif/Times, 320 dpi, pdf type 42).")

In [ ]:
#  INPUT: set H5_PATH (single) OR H5_INPUT (dir / glob / list)  
H5_PATH  = "/home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/runs/put_black_bowl_on_plate/put_black_bowl_on_plate__0498e398b31d.h5"
# H5_INPUT = "/home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/runs/put_black_bowl_on_plate"
# H5_INPUT = "/home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/runs/put_black_bowl_on_plate/*.h5"
# H5_INPUT = ["a.h5", "b.h5"]

def resolve_h5_inputs():
    raw = globals().get("H5_INPUT", None)
    if raw is None:
        p = globals().get("H5_PATH", None)
        if p is None: raise ValueError("Set H5_PATH (single) or H5_INPUT (batch).")
        paths = [Path(p)]
    elif isinstance(raw, (list, tuple)):
        paths = [Path(x) for x in raw]
    elif isinstance(raw, (str, Path)) and any(c in str(raw) for c in "*?["):
        paths = [Path(x) for x in glob.glob(str(raw))]
    else:
        q = Path(raw)
        paths = sorted(q.glob("*.h5")) if q.is_dir() else [q]
    paths = [p for p in paths if p.suffix == ".h5" and p.is_file()]
    if not paths: raise FileNotFoundError(f"No .h5 from input {raw!r}")
    return sorted(set(paths))

H5_PATHS  = resolve_h5_inputs()
IS_BATCH  = len(H5_PATHS) > 1
H5_PATH   = str(H5_PATHS[0])                    # representative
TAG       = H5_PATHS[0].parent.name if IS_BATCH else H5_PATHS[0].stem

from surena_vla.paths import OUTPUTS_ROOT
OUT_ROOT  = Path(os.environ.get("SURENA_EVALUATION_RESULTS_DIR", OUTPUTS_ROOT / "evaluation_results"))
FIG_DIR, TABLE_DIR, VIDEO_DIR = OUT_ROOT / "figures", OUT_ROOT / "tables", OUT_ROOT / "videos"
for d in (FIG_DIR, TABLE_DIR, VIDEO_DIR): d.mkdir(parents=True, exist_ok=True)

def savefig(fig, name, subdir=None):
    base = FIG_DIR / subdir if subdir else FIG_DIR
    base.mkdir(parents=True, exist_ok=True)
    for ext in ("png", "pdf"):          # pdf for the paper, png for quick view
        fig.savefig(base / f"{name}.{ext}", bbox_inches="tight")
    print(f"[fig] {base / name}.png|.pdf")
    return base / f"{name}.png"

print(f"{'BATCH' if IS_BATCH else 'SINGLE'} n={len(H5_PATHS)} tag={TAG!r}")
for p in H5_PATHS: print(f"  {p}")
print(f"figures -> {FIG_DIR}\ntables  -> {TABLE_DIR}\nvideos  -> {VIDEO_DIR}")


## 2. HDF5 inspection (representative file)

In [ ]:
def _short(v, m=100):
    s = v.decode() if isinstance(v, bytes) else str(v)
    return s if len(s) <= m else s[:m] + "..."

def inspect_h5(path, max_attr_len=100, max_items=400):
    root = {"groups": [], "datasets": [], "root_attrs": {}}
    with h5py.File(path, "r") as f:
        root["root_attrs"] = {k: _short(v, max_attr_len) for k, v in f.attrs.items()}
        print(f"/  root attrs: {list(f.attrs)}")
        def vis(name, obj):
            ind = "    " * name.count("/")
            if isinstance(obj, h5py.Group):
                root["groups"].append(name); print(f"{ind}[G] {name}/")
                for k, v in obj.attrs.items(): print(f"{ind}    @{k} = {_short(v)}")
            else:
                root["datasets"].append({"path": name, "shape": obj.shape, "dtype": str(obj.dtype)})
                print(f"{ind}[D] {name}  shape={obj.shape} dtype={obj.dtype}")
                for k, v in obj.attrs.items(): print(f"{ind}    @{k} = {_short(v)}")
        f.visititems(vis)
    return root

h5_summary = inspect_h5(H5_PATH)
print(f"\n{len(h5_summary['groups'])} groups, {len(h5_summary['datasets'])} datasets")


## 3. Signal detection (name-based, schema-aware)

In [ ]:
SIGNAL_PATTERNS = {
    "vla_actions":     [r"exec_action", r"raw_action"],  # executed action is canonical
    "joint_commands":  [r"q_cmd\b", r"q_des"],
    "joint_positions": [r"q_actual\b", r"\bqpos$"],
    "joint_velocities":[r"/qvel\b", r"\bqvel$"],
    "joint_torques":   [r"actuator_force", r"qfrc_applied"],
    "eef_pos":         [r"eef_pos\b(?!_cmd|_target)", r"ee_?pos"],
    "eef_quat":        [r"eef_rpy\b", r"eef.*quat"],
    "eef_pos_cmd":     [r"commanded_target_pos"],
    "eef_quat_cmd":    [r"commanded_target_quat"],
    "ik_status":       [r"ik_status\b", r"ik_stage\b", r"escalation_reason"],
    "ik_success":      [r"ik_ok\b", r"ik_accepted\b", r"ik_feasible\b"],
    "ik_pos_error":    [r"ik_pos_err\b"],
    "ik_rot_error":    [r"ik_rot_err\b"],
    "ik_solve_time":   [r"ik_solve_ms\b"],
    "ik_iterations":   [r"ik_attempt_count\b"],
    "collisions":      [r"collision_hard", r"collision_min_dist"],
    "gripper_state":   [r"sticky_state\b"],
    "gripper_command": [r"sticky_.*command"],
    "timestamps":      [r"ticks/t_wall\b", r"vla_steps/t_wall\b"],
    "rewards":         [r"^reward"],
    "success":         [r"^success$"],
    "task_name":       [r"preset_name", r"task_?name"],
    "control_freq":    [r"ctrl_freq_hz", r"control_?freq"],
}
ATTR_SIGNALS = {"success", "task_name", "control_freq"}

def detect_signals(summary):
    det = defaultdict(list)
    for d in summary["datasets"]:
        leaf = d["path"].lower()
        for sig, pats in SIGNAL_PATTERNS.items():
            if sig in ATTR_SIGNALS: continue
            if any(re.search(p, leaf) for p in pats): det[sig].append(d["path"])
    for sig in ATTR_SIGNALS:
        for pat in SIGNAL_PATTERNS[sig]:
            hit = next((f"@{k}" for k in summary.get("root_attrs", {}) if re.search(pat, k.lower())), None)
            if hit: det[sig].append(hit); break
    return dict(det)

detected = detect_signals(h5_summary)
for s, ps in sorted(detected.items()): print(f"  {s:18s} -> {ps}")
missing = [s for s in SIGNAL_PATTERNS if s not in detected]
if missing:
    print("[INFO] unmatched:", ", ".join(missing), "(rewards expected absent in this schema)")


## 4. Extraction — single & batch

In [ ]:
ATTR_BACKED  = {"success", "task_name", "control_freq"}
ATTR_SOURCE  = {"success": "success", "task_name": "preset_name", "control_freq": "ctrl_freq_hz"}
REQUIRED     = [s for s in SIGNAL_PATTERNS if s != "rewards"]
OVERRIDES: dict[str, str] = {}   # e.g. {"eef_pos": "vla_steps/eef_pos_after"}

def load_episode(path):
    path = Path(path)
    src, ep = {}, {}
    with h5py.File(path, "r") as f:
        summ = {"datasets": [{"path": k} for k in _walk_paths(f)], "root_attrs": dict(f.attrs)}
        det  = detect_signals(summ)
        for key in REQUIRED:
            if key == "vla_actions" and key not in OVERRIDES:
                p = next((name for name in ("vla_steps/exec_action", "vla_steps/raw_action") if name in f), None)
            else:
                p = OVERRIDES.get(key) or (det.get(key) or [None])[0]
            if key in ATTR_BACKED:
                ak = ATTR_SOURCE[key]
                v = f.attrs[ak] if ak in f.attrs else None
                if isinstance(v, bytes): v = v.decode()
                if isinstance(v, np.generic): v = v.item()
                ep[key] = v; src[key] = f"@{ak}"; continue
            if p is None: ep[key] = None; continue
            if p.startswith("@"):
                v = f.attrs[p[1:]]; ep[key] = v.decode() if isinstance(v, bytes) else v
            else:
                obj = f[p]; ep[key] = obj[()] if isinstance(obj, h5py.Dataset) else None
            src[key] = p
        ep["_attrs"] = {k: (v.decode() if isinstance(v, bytes) else (v.item() if isinstance(v, np.generic) else v))
                        for k, v in f.attrs.items()}
    ep["_path"], ep["_stem"] = str(path), path.stem
    return ep, src

def _walk_paths(f):
    out = []
    f.visititems(lambda n, o: out.append(n) if isinstance(o, h5py.Dataset) else None)
    return out

episode, SIGNAL_SOURCES = load_episode(H5_PATHS[0])
print(f"representative: {episode['_stem']}")
for k, v in SIGNAL_SOURCES.items():
    val = episode.get(k)
    d = f"ndarray{np.shape(val)}" if isinstance(val, np.ndarray) else repr(val)[:40]
    print(f"  {k:18s} <- {str(v):34s} {d}")

episodes, sources_list = [], []
for p in H5_PATHS:
    try:
        ep, _ = load_episode(p); episodes.append(ep)
    except Exception as e:
        print(f"[WARN] {p.name}: {e}")
sources_list = [SIGNAL_SOURCES] + [SIGNAL_SOURCES] * (len(episodes) - 1)
print(f"loaded {len(episodes)}/{len(H5_PATHS)}")


## 5. Time base

uniform `physics_dt` grid — wall clocks only for cross-check

In [ ]:
def build_timebase(ep):
    attrs = ep.get("_attrs", {})
    dt = attrs.get("physics_dt"); dt = float(dt) if dt is not None else None
    cf = attrs.get("ctrl_freq_hz"); cf = float(cf) if cf is not None else None
    N = None
    for k in ("joint_positions", "eef_pos", "joint_commands"):
        v = ep.get(k)
        if v is not None and hasattr(v, "shape"): N = int(v.shape[0]); break
    if dt and N:
        t = np.arange(N) * dt
    elif ep.get("timestamps") is not None:
        w = np.asarray(ep["timestamps"], float); t = w - w[0]
    else:
        t = np.arange(N or 0, dtype=float)
    ep["_t"], ep["_dt"], ep["_cf"] = t, dt, cf
    ep["_dur"] = float(t[-1]) if len(t) else 0.0
    ep["_n_ticks"] = N
    return t

for ep in episodes: build_timebase(ep)
episode = episodes[0]   # same object as the list, so _t/_dur are set
t = episode["_t"]; DURATION_S = episode["_dur"]; physics_dt = episode["_dt"]; N_TICKS = episode["_n_ticks"]
print(f"representative: {N_TICKS} ticks, {DURATION_S:.2f} s sim, physics_dt={physics_dt}, ctrl={episode['_cf']} Hz")
print(f"batch ticks: {[ep['_n_ticks'] for ep in episodes]}")


## 6. Summary dashboard

In [ ]:
def get_task(ep):
    v = ep.get("task_name") or ep["_attrs"].get("preset_name") or "unknown"
    return str(v)

def get_success(ep):
    s = ep.get("success", ep["_attrs"].get("success"))
    if s is None: return None
    if isinstance(s, str): return s.lower() in ("1", "true", "success", "yes")
    return bool(np.atleast_1d(s)[-1])

print("=" * 56); print("EPISODE SUMMARY (representative)"); print("=" * 56)
A = episode["_attrs"]
info = dict(task=get_task(episode), success=get_success(episode),
           ticks=N_TICKS, duration_sim_s=round(DURATION_S, 2),
           duration_wall_s=round(float(A["completion_time_wall_s"]), 2) if "completion_time_wall_s" in A else None,
           ctrl_hz=episode["_cf"], sim_hz=A.get("sim_freq_hz"),
           stop_reason=A.get("stop_reason", "?"))
for k, v in info.items(): print(f"{k:18s}: {v}")

if IS_BATCH:
    succ = [get_success(e) for e in episodes]
    dur  = np.array([e["_dur"] for e in episodes])
    m, s, ci, n = mean_std_ci(dur)
    rate = np.mean([x for x in succ if x is not None]) if any(x is not None for x in succ) else None
    print("-" * 56)
    print(f"BATCH n={len(episodes)}  duration {m:.2f} ± {s:.2f} s (std) ± {ci:.2f} s (95% CI)")
    if rate is not None:
        print(f"success rate {rate:.0%} ({sum(x for x in succ if x is not None)}/{len(succ)})")
    print(f"tasks: {Counter(get_task(e) for e in episodes)}")
    print(f"stop reasons: {Counter(str(e['_attrs'].get('stop_reason', '?')) for e in episodes)}")


## 7. EEF trajectory & joint trajectories

In [ ]:
pos = np.asarray(episode["eef_pos"]) if episode.get("eef_pos") is not None else None
if pos is not None:
    # Separate figures: no static subfigure layouts.
    fig = plt.figure(figsize=FIG_SQUARE)
    ax3 = fig.add_subplot(111, projection="3d")
    ax3.plot(pos[:, 0], pos[:, 1], pos[:, 2], color="#4477AA", lw=1.6)
    ax3.scatter(*pos[0], color="#228833", s=40, edgecolors="w", lw=0.6, label="start")
    ax3.scatter(*pos[-1], color="#C92A2A", s=40, edgecolors="w", lw=0.6, label="end")
    ax3.set_xlabel("x [m]", fontsize=9); ax3.set_ylabel("y [m]", fontsize=9); ax3.set_zlabel("z [m]", fontsize=9)
    ax3.legend(fontsize=8); ax3.tick_params(labelsize=7)
    savefig(fig, f"{TAG}__eef_trajectory_3d"); plt.show()

    fig, ax = plt.subplots(figsize=FIG_SQUARE)
    tt = t[:len(pos)]
    for i, (lab, c) in enumerate(zip("xyz", XYZ_COLORS)):
        ax.plot(tt, pos[:, i], label=lab, color=c, lw=1.5)
    _style_ax(ax, xlabel="time [s]", ylabel="EEF position [m]")
    savefig(fig, f"{TAG}__eef_position_time"); plt.show()
else:
    print("[WARN] eef_pos missing")

JN = ["pitch", "roll", "elbow", "forearm_roll", "forearm_link", "hand_pitch", "hand_roll"]
if episode.get("joint_positions") is not None:
    q = np.asarray(episode["joint_positions"])
    qc = np.asarray(episode["joint_commands"]) if episode.get("joint_commands") is not None else None
    for i in range(q.shape[1]):
        name = JN[i] if i < len(JN) else f"j{i}"
        fig, ax = plt.subplots(figsize=FIG_SQUARE)
        ax.plot(t[:len(q)], q[:, i], color="#4477AA", lw=1.1, label="measured")
        if qc is not None and len(qc) == len(q):
            ax.plot(t[:len(qc)], qc[:, i], color="#EE6677", lw=1.0, ls="--", label="commanded")
        _style_ax(ax, xlabel="time [s]", ylabel=f"{name} angle [rad]")
        savefig(fig, f"{TAG}__joint_{i:02d}_{name}"); plt.show()
    if qc is not None and qc.shape == q.shape:
        err = np.abs(qc - q).mean(axis=0)
        print("mean |commanded-measured| [rad]:", {JN[i] if i < len(JN) else f"j{i}": round(float(err[i]), 4) for i in range(len(err))})


## 8. Video from the episode log

Frames come from the file's own JPEG datasets (`video_frames`, `keyframes`) — decoded with `surena_vla.telemetry.codec.decode_jpeg`, **no re-simulation**:

* **Mode A — plain MP4** (`videos/<stem>.mp4`, 60 fps): all `video_frames`.
* **Mode B — animated frame | action history** (`<stem>__history.mp4` + inline HTML, 8 fps): 1×2 panel — frame + 7 action curves growing per VLA step.

The action history uses `vla_steps/exec_action`.


In [ ]:
from surena_vla.telemetry.codec import decode_jpeg

def decode_frames_from_h5(path):
    path = Path(path)
    with h5py.File(path, "r") as f:
        out = {"keyframes": [], "video_frames": []}
        if "keyframes" in f:
            out["keyframes"] = [decode_jpeg(b) for b in f["keyframes"][()]]
        if "video_frames" in f:
            out["video_frames"] = [decode_jpeg(b) for b in f["video_frames"][()]]
        out["task"] = str(f.attrs.get("preset_name", path.stem))
        if "vla_steps/exec_action" in f:
            out["actions"] = np.asarray(f["vla_steps/exec_action"][()])
            out["action_source"] = "vla_steps/exec_action"
        elif "vla_steps/raw_action" in f:
            out["actions"] = np.asarray(f["vla_steps/raw_action"][()])
            out["action_source"] = "vla_steps/raw_action"
    return out

def save_mp4(frames, out_path, fps=60):
    import imageio.v2 as imageio
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    with imageio.get_writer(str(out_path), fps=fps, codec="libx264",
                            quality=8, macro_block_size=1) as w:
        for fr in frames:
            fr = np.asarray(fr)
            if fr.dtype != np.uint8: fr = np.clip(fr, 0, 255).astype(np.uint8)
            w.append_data(fr)
    print(f"[video] {out_path} ({len(frames)} frames @ {fps} fps)")
    return out_path

def _hist_setup(frames, actions, task, figsize):
    """Build fig/upd shared by HTML + MP4 paths (mimics §8 cell 65db6e72)."""
    acts = np.asarray(actions, float)[:, :7]
    n = len(acts)
    if len(frames) != n and len(frames) > n:
        frames = [frames[i] for i in np.linspace(0, len(frames) - 1, n, dtype=int)]
    frames = list(frames[:n]) or [np.zeros((224, 224, 3), np.uint8)] * n
    if len(frames) < n:
        frames = frames + [frames[-1]] * (n - len(frames))
    fig, (ax_f, ax_a) = plt.subplots(1, 2, figsize=figsize)
    im = ax_f.imshow(frames[0]); ax_f.set_title("episode"); ax_f.axis("off")
    txt = ax_f.text(5, 15, "", color="w", fontsize=10, bbox=dict(facecolor="black", alpha=0.55))
    lines = [ax_a.plot([], [], label=l, color=c, lw=1.5)[0]
             for l, c in zip(VLA_LABELS_7, VLA_COLORS_7)]
    lo, hi = float(np.nanmin(acts)), float(np.nanmax(acts))
    pad = max(0.1, 0.05 * (hi - lo + 1e-9))
    ax_a.set_xlim(0, max(1, n - 1)); ax_a.set_ylim(lo - pad, hi + pad)
    ax_a.set_title("Action History"); ax_a.set_xlabel("VLA step [count]"); ax_a.set_ylabel("normalized action [1]")
    ax_a.axhline(0, color="k", lw=0.6, ls="--", alpha=0.6)
    ax_a.legend(fontsize=7, ncol=2, loc="upper right")
    def upd(k):
        im.set_data(frames[k]); txt.set_text(f"Step {k+1}/{n}")
        x = np.arange(k + 1)
        for j, ln in enumerate(lines): ln.set_data(x, acts[:k + 1, j])
        return [im, txt] + lines
    plt.tight_layout()
    return fig, upd, n

def show_action_history_html(frames, actions, task="", fps=8):
    """Mode B inline (cf. show_episode_action_history)."""
    fig, upd, n = _hist_setup(frames, actions, task, figsize=(12.5, 5.2))
    anim = animation.FuncAnimation(fig, upd, frames=n, interval=int(1000 / fps), blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

def save_action_history_mp4(frames, actions, out_path, task="", fps=8):
    """Mode B saved MP4."""
    fig, upd, n = _hist_setup(frames, actions, task, figsize=(12.5, 5.2))
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    writer = animation.FFMpegWriter(fps=fps, metadata=dict(artist="surena-vla"))
    with writer.saving(fig, str(out_path), dpi=120):
        for k in range(n): upd(k); writer.grab_frame()
    plt.close(fig)
    print(f"[video] {out_path} ({n} steps @ {fps} fps)")
    return out_path

def export_videos(path):
    d = decode_frames_from_h5(path)
    res = {}
    if d["video_frames"]:
        res["plain"] = save_mp4(d["video_frames"], VIDEO_DIR / f"{path.stem}.mp4", fps=60)
    else:
        print(f"[WARN] {path.name}: no video_frames")
    acts = d.get("actions")
    if acts is not None:
        print(f"[video] action history source: {d.get('action_source', 'unknown')}")
    if acts is not None and len(acts) and (d["keyframes"] or d["video_frames"]):
        hframes = d["keyframes"] or d["video_frames"]
        res["history"] = save_action_history_mp4(hframes, acts, VIDEO_DIR / f"{path.stem}__history.mp4",
                                                 task=d["task"], fps=8)
    return res

videos_repr = export_videos(H5_PATHS[0])
if videos_repr.get("history") is not None:
    d = decode_frames_from_h5(H5_PATHS[0])
    display(show_action_history_html(d["keyframes"] or d["video_frames"], d["actions"], task=d["task"], fps=8))
if IS_BATCH:
    for p in H5_PATHS[1:]: export_videos(p)
    print(f"all videos -> {VIDEO_DIR}")


## 9. Motion quality — Cartesian tracking

In [ ]:
def cartesian_metrics(ep):
    """commanded_target vs achieved (VLA-step granularity; eef_pos_after preferred)."""
    out = {}
    with h5py.File(ep["_path"], "r") as f:
        pc = np.asarray(f["vla_steps/commanded_target_pos"][()]) if "vla_steps/commanded_target_pos" in f else None
        pa = np.asarray(f["vla_steps/eef_pos_after"][()]) if "vla_steps/eef_pos_after" in f else None
        qc = np.asarray(f["vla_steps/commanded_target_quat"][()]) if "vla_steps/commanded_target_quat" in f else None
        qa_rpy = np.asarray(f["vla_steps/eef_rpy_after"][()]) if "vla_steps/eef_rpy_after" in f else None
        te_p = np.asarray(f["vla_steps/tracking_pos_err"][()], float) if "vla_steps/tracking_pos_err" in f else None
        te_r = np.asarray(f["vla_steps/tracking_rot_err"][()], float) if "vla_steps/tracking_rot_err" in f else None
    if pc is not None and pa is not None:
        n = min(len(pc), len(pa))
        e = np.linalg.norm(pc[:n] - pa[:n], axis=-1)
        out.update(mean_pos_err_m=float(e.mean()), max_pos_err_m=float(e.max()), rms_pos_err_m=float(np.sqrt((e**2).mean())))
    if qc is not None and qc.shape[-1] == 4 and qa_rpy is not None:
        from surena_vla.telemetry.codec import quat_wxyz_to_euler as q2r
        qc_rpy = np.array([q2r(q) for q in qc[:min(len(qc), len(qa_rpy))]])
        ae = np.degrees(np.linalg.norm(qc_rpy - qa_rpy[:len(qc_rpy)], axis=-1))
        out.update(mean_rot_err_deg=float(ae.mean()), max_rot_err_deg=float(ae.max()))
    elif te_r is not None:
        out.update(mean_rot_err_deg=float(np.degrees(te_r).mean()), max_rot_err_deg=float(np.degrees(te_r).max()))
    if te_p is not None and "mean_pos_err_m" not in out:
        out.update(mean_pos_err_m=float(te_p.mean()), max_pos_err_m=float(te_p.max()))
    return out

CART = cartesian_metrics(episode)
print("representative:", {k: round(v, 4) for k, v in CART.items()})
if IS_BATCH:
    rows = [cartesian_metrics(e) for e in episodes]
    for k in ("mean_pos_err_m", "max_pos_err_m", "rms_pos_err_m", "mean_rot_err_deg", "max_rot_err_deg"):
        v = np.array([r.get(k, np.nan) for r in rows], float)
        if np.all(np.isnan(v)): continue
        m, s, ci, n = mean_std_ci(v)
        print(f"batch {k:18s} {m:.4f} ± {s:.4f} (std) ± {ci:.4f} (95% CI, n={n})")


## 10. Smoothness — velocity / acceleration / jerk

Startup data before `t = 1 s` is excluded from both the figures and reported smoothness metrics.
Each quantity is exported as its own figure.


In [ ]:
SMOOTHNESS_START_S = 1.0  # discard startup transient; plots and metrics begin in the second second

def _deriv(x, tt):
    return np.gradient(np.asarray(x, float), np.asarray(tt, float), axis=0)

def _after_start(tt, start_s=SMOOTHNESS_START_S):
    tt = np.asarray(tt, float)
    mask = tt >= start_s
    return mask if mask.any() else np.ones(len(tt), dtype=bool)

def smoothness_metrics(ep):
    out = {}
    if ep.get("eef_pos") is None or len(ep["_t"]) < 3: return out
    pos = np.asarray(ep["eef_pos"]); tt = ep["_t"][:len(pos)]
    v = _deriv(pos, tt); a = _deriv(v, tt); j = _deriv(a, tt)
    vm, am, jm = (np.linalg.norm(x, axis=-1) for x in (v, a, j))
    keep = _after_start(tt)
    tt, vm, am, jm = tt[keep], vm[keep], am[keep], jm[keep]
    out.update(mean_vel=float(vm.mean()), max_vel=float(vm.max()),
               mean_acc=float(am.mean()), max_acc=float(am.max()),
               rms_jerk=float(np.sqrt((jm**2).mean())), _t=tt, _v=vm, _a=am, _j=jm)
    if ep.get("joint_positions") is not None:
        q = np.asarray(ep["joint_positions"]); tq = ep["_t"][:len(q)]
        qd = _deriv(q, tq); qdd = _deriv(qd, tq); qddd = _deriv(qdd, tq)
        qkeep = _after_start(tq)
        out.update(joint_max_vel=float(np.abs(qd[qkeep]).max()),
                   joint_rms_jerk=float(np.sqrt((qddd[qkeep]**2).mean())))
    return out

SM = smoothness_metrics(episode)
if SM:
    for data, ylabel, color, suffix in (
        (SM["_v"], "EEF speed [m/s]", "#4477AA", "velocity"),
        (SM["_a"], "EEF acceleration [m/s²]", "#EE6677", "acceleration"),
        (SM["_j"], "EEF jerk [m/s³]", "#228833", "jerk"),
    ):
        fig, ax = plt.subplots(figsize=FIG_SQUARE)
        ax.plot(SM["_t"], data, color=color, lw=1.1)
        _style_ax(ax, xlabel="time [s]", ylabel=ylabel)
        savefig(fig, f"{TAG}__eef_{suffix}"); plt.show()
    print(f"representative (t >= {SMOOTHNESS_START_S:g} s):",
          {k: round(v, 4) for k, v in SM.items() if not k.startswith("_")})

if IS_BATCH:
    sms = [smoothness_metrics(e) for e in episodes]
    for k in ("mean_vel", "max_vel", "mean_acc", "max_acc", "rms_jerk", "joint_max_vel", "joint_rms_jerk"):
        v = np.array([d.get(k, np.nan) for d in sms], float)
        if np.all(np.isnan(v)): continue
        m, s, ci, n = mean_std_ci(v)
        print(f"batch {k:16s} {m:.4f} ± {s:.4f} (std) ± {ci:.4f} (95% CI; t >= {SMOOTHNESS_START_S:g} s)")
    stack = []
    for d in sms:
        if "_v" not in d: continue
        x = np.linspace(0, 1, len(d["_v"])); xi = np.linspace(0, 1, 200)
        stack.append(np.interp(xi, x, d["_v"]))
    if stack:
        A = np.vstack(stack)
        mean, std, ci, n = mean_std_ci(A, axis=0)
        end_s = max((d["_t"][-1] for d in sms if "_t" in d and len(d["_t"])), default=SMOOTHNESS_START_S)
        xi = np.linspace(SMOOTHNESS_START_S, end_s, 200)
        fig, ax = plt.subplots(figsize=FIG_SQUARE)
        ax.plot(xi, mean, color="#4477AA", lw=1.8, label="mean EEF speed")
        ax.fill_between(xi, mean - ci, mean + ci, color="#4477AA", alpha=CI_ALPHA, label="95% CI")
        _style_ax(ax, xlabel="time [s]", ylabel="EEF speed [m/s]")
        savefig(fig, "batch__eef_speed_ci", subdir="batch"); plt.show()


## 11. Effort

In [ ]:
def effort_metrics(ep):
    if ep.get("joint_torques") is None or ep.get("joint_velocities") is None: return {}
    tau = np.asarray(ep["joint_torques"]); qd = np.asarray(ep["joint_velocities"])
    n = min(len(tau), len(qd)); tt = ep["_t"][:n]
    P = np.abs(tau[:n] * qd[:n]).sum(axis=-1)
    return dict(effort_J=float(np.trapz(P, tt)) if len(tt) > 1 else 0.0,
                mean_abs_tau=float(np.abs(tau[:n]).mean()),
                rms_tau=float(np.sqrt((tau[:n]**2).mean())))

EFF = effort_metrics(episode)
print("representative:", {k: round(v, 3) for k, v in EFF.items()} or "(unavailable)")
if IS_BATCH:
    rows = [effort_metrics(e) for e in episodes]
    for k in ("effort_J", "mean_abs_tau", "rms_tau"):
        v = np.array([r.get(k, np.nan) for r in rows], float)
        if np.all(np.isnan(v)): continue
        m, s, ci, n = mean_std_ci(v)
        print(f"batch {k:12s} {m:.3f} ± {s:.3f} (std) ± {ci:.3f} (95% CI)")


## 12. IK performance — separate solution-rate, error, and latency figures

IK stages are read directly from `vla_steps/ik_stage`; the root count attribute is only a fallback.


In [ ]:
def _decode_text_array(values):
    return np.array([x.decode() if isinstance(x, bytes) else str(x) for x in np.atleast_1d(values)])

def _stage_rates(stages):
    stages = _decode_text_array(stages)
    lower = np.char.lower(stages.astype(str))
    full = np.char.find(lower, "full") >= 0
    pos = (np.char.find(lower, "position") >= 0) | (np.char.find(lower, "pos_only") >= 0)
    fail = ((np.char.find(lower, "fail") >= 0) | (np.char.find(lower, "hold") >= 0))
    # Unknown names remain visible in stage_counts and are included in the denominator.
    total = max(len(stages), 1)
    counts = {str(k): int(v) for k, v in zip(*np.unique(stages, return_counts=True))}
    return dict(ik_full_pct=100 * full.sum() / total,
                ik_pos_only_pct=100 * pos.sum() / total,
                ik_failed_pct=100 * fail.sum() / total,
                stage_counts=counts, _stage=stages)

def ik_metrics(ep):
    out = {}
    with h5py.File(ep["_path"], "r") as f:
        g = f.get("vla_steps")
        if g is not None and "ik_stage" in g:
            out.update(_stage_rates(g["ik_stage"][()]))
        elif "ik_stage_counts_json" in f.attrs:
            raw = f.attrs["ik_stage_counts_json"]
            sc = json.loads(raw if isinstance(raw, str) else raw.decode())
            expanded = [name for name, count in sc.items() for _ in range(int(count))]
            out.update(_stage_rates(expanded))
            out.pop("_stage", None)
        if g is not None:
            for key, dst in (("ik_pos_err", "_pos_err"), ("ik_rot_err", "_rot_err"), ("ik_solve_ms", "_solve_ms")):
                if key in g: out[dst] = np.asarray(g[key][()], float)
    for src, metric in (("_pos_err", "ik_mean_pos_err"), ("_rot_err", "ik_mean_rot_err"), ("_solve_ms", "ik_mean_solve_ms")):
        if src in out and np.isfinite(out[src]).any(): out[metric] = float(np.nanmean(out[src]))
    return out

def _ik_pie(ax, vals, title):
    labels = ["full-pose", "position-only", "failed/hold"]
    colors = [IK_PIE_COLORS["full"], IK_PIE_COLORS["pos"], IK_PIE_COLORS["fail"]]
    triples = [(v, label, color) for v, label, color in zip(vals, labels, colors) if v > 0.5]
    if not triples:
        ax.text(0.5, 0.5, "no classified IK stages\n(see stage counts)", ha="center", va="center"); return
    values, labels, colors = zip(*triples)
    _, _, auto = ax.pie(values, labels=labels, colors=colors, startangle=90, counterclock=False,
                        autopct=lambda p: f"{p:.1f}%" if p >= 3 else "", pctdistance=0.75,
                        wedgeprops=dict(width=0.42, edgecolor="white", linewidth=1.5),
                        textprops=dict(fontsize=9))
    for label in auto: label.set_color("white"); label.set_fontweight("bold"); label.set_fontsize(8)

IK = ik_metrics(episode)
_ik_src = "vla_steps/ik_stage" if "_stage" in IK else ("@ik_stage_counts_json" if "stage_counts" in IK else "NOT FOUND")
print(f"IK source: {_ik_src} | stage_counts = {IK.get('stage_counts', {})}")
if IK:
    fig, ax = plt.subplots(figsize=FIG_SQ_SMALL)
    _ik_pie(ax, (IK.get("ik_full_pct", 0), IK.get("ik_pos_only_pct", 0), IK.get("ik_failed_pct", 0)),
            "IK solution rates (representative)")
    savefig(fig, f"{TAG}__ik_pie"); plt.show()
    print("representative:", {k: (round(v, 2) if isinstance(v, float) else v)
                              for k, v in IK.items() if not k.startswith("_")})
    if "stage_counts" in IK:
        fig, ax = plt.subplots(figsize=FIG_HIST)
        labels = list(IK["stage_counts"].keys()); counts = [IK["stage_counts"][label] for label in labels]
        ax.barh(labels, counts, color="#4477AA", edgecolor="white")
        for i, count in enumerate(counts): ax.text(count, i, f" {count}", va="center", fontsize=8)
        _style_ax(ax, xlabel="IK solutions [count]", legend=False, yticks5=False)
        savefig(fig, f"{TAG}__ik_stages"); plt.show()
    for key, color, ylabel, suffix in (
        ("_pos_err", "#4477AA", "IK position error [m]", "pos_err"),
        ("_rot_err", "#66CCEE", "IK rotation error [rad]", "rot_err"),
        ("_solve_ms", "#228833", "IK solve time [ms]", "solve_ms"),
    ):
        if key not in IK: continue
        fig, ax = plt.subplots(figsize=FIG_SQUARE)
        ax.plot(IK[key], color=color, lw=1.3)
        _style_ax(ax, xlabel="VLA step [count]", ylabel=ylabel, legend=False)
        savefig(fig, f"{TAG}__ik_{suffix}"); plt.show()

if IS_BATCH:
    iks = [ik_metrics(e) for e in episodes]
    keys = ("ik_full_pct", "ik_pos_only_pct", "ik_failed_pct")
    raw_means = [np.nanmean([d.get(k, np.nan) for d in iks]) for k in keys]
    total = np.nansum(raw_means)
    vals = tuple(100 * x / total for x in raw_means) if total > 0 else (0, 0, 0)
    fig, ax = plt.subplots(figsize=FIG_SQ_SMALL)
    _ik_pie(ax, vals, f"batch mean (n={len(iks)})")
    savefig(fig, "batch__ik_rates_pie", subdir="batch"); plt.show()

    labels = ["full-pose", "position-only", "failed/hold"]
    colors = [IK_PIE_COLORS["full"], IK_PIE_COLORS["pos"], IK_PIE_COLORS["fail"]]
    means, cis = [], []
    for k in keys:
        values = np.array([d.get(k, np.nan) for d in iks], float)
        m, std, ci, n = mean_std_ci(values)
        means.append(m if np.isfinite(m) else 0.0); cis.append(ci if np.isfinite(ci) else 0.0)
    x = np.arange(3)
    fig, ax = plt.subplots(figsize=FIG_SQUARE)
    ax.bar(x, means, yerr=cis, capsize=6, color=colors, edgecolor="white",
           error_kw=dict(elinewidth=1.2, capthick=1.2))
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9); ax.set_ylim(0, 105)
    _style_ax(ax, ylabel="IK solutions [%]", legend=False)
    savefig(fig, "batch__ik_rates_ci", subdir="batch"); plt.show()

    for k in (*keys, "ik_mean_pos_err", "ik_mean_rot_err", "ik_mean_solve_ms"):
        values = np.array([d.get(k, np.nan) for d in iks], float)
        if np.all(np.isnan(values)): continue
        m, std, ci, n = mean_std_ci(values)
        print(f"batch {k:20s} {m:.4f} ± {std:.4f} (std) ± {ci:.4f} (95% CI)")
    for key, color, ylabel, fname in (
        ("_pos_err", "#4477AA", "IK position error [m]", "batch__ik_pos_err_ci"),
        ("_rot_err", "#66CCEE", "IK rotation error [rad]", "batch__ik_rot_err_ci"),
        ("_solve_ms", "#228833", "IK solve time [ms]", "batch__ik_solve_ms_ci"),
    ):
        arrs = [d[key] for d in iks if key in d]
        if not arrs: continue
        length = max(len(a) for a in arrs)
        padded = np.full((len(arrs), length), np.nan)
        for i, a in enumerate(arrs): padded[i, :len(a)] = a
        mean = np.nanmean(padded, axis=0); std = np.nanstd(padded, axis=0, ddof=1)
        nobs = np.sum(~np.isnan(padded), axis=0); sem = std / np.sqrt(np.maximum(nobs, 1))
        ci = np.array([ci95_factor(k) if k > 1 else np.nan for k in nobs]) * sem
        x = np.arange(length)
        fig, ax = plt.subplots(figsize=FIG_SQUARE)
        ax.plot(x, mean, color=color, lw=1.8, label="mean")
        ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=CI_ALPHA, label="95% CI")
        _style_ax(ax, xlabel="VLA step [count]", ylabel=ylabel)
        savefig(fig, fname, subdir="batch"); plt.show()


## 13. VLA actions

* **separate**: (a) dx/dy/dz together, (b) droll/dpitch/dyaw together, (c) gripper alone — each is its own square-ish figure.
* **combined**: all 7 in one figure.
* The canonical source is `vla_steps/exec_action`, with `raw_action` used only as a fallback.
* Batch: NaN-padded ragged horizons → **mean ± 95% CI** band versions of both, + `batch_actions_padded.npz`.


In [ ]:
def load_actions(ep):
    a = ep.get("vla_actions")
    return np.asarray(a, float)[:, :7] if a is not None else None

def plot_actions_split(acts, out_prefix, subdir=None):
    x = np.arange(len(acts))
    # (a) translation
    fig, ax = plt.subplots(figsize=FIG_SQUARE)
    for i, (lab, c) in enumerate(zip(("dx", "dy", "dz"), XYZ_COLORS)):
        ax.plot(x, acts[:, i], label=lab, color=c, lw=1.6)
    ax.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    _style_ax(ax, xlabel="VLA step [count]", ylabel="normalized action [1]")
    savefig(fig, f"{out_prefix}__actions_xyz", subdir=subdir); plt.show()
    # (b) rotation
    fig, ax = plt.subplots(figsize=FIG_SQUARE)
    for i, (lab, c) in enumerate(zip(("droll", "dpitch", "dyaw"), RPY_COLORS)):
        ax.plot(x, acts[:, 3 + i], label=lab, color=c, lw=1.6)
    ax.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    _style_ax(ax, xlabel="VLA step [count]", ylabel="normalized action [1]")
    savefig(fig, f"{out_prefix}__actions_rpy", subdir=subdir); plt.show()
    # (c) gripper
    fig, ax = plt.subplots(figsize=FIG_SQ_SMALL)
    ax.plot(x, acts[:, 6], color=GRIP_COLOR, lw=1.8)
    ax.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    ax.axhline(0.5, color=GRIP_COLOR, lw=0.8, ls=":", alpha=0.7, label="0.5 threshold")
    ax.set_ylim(min(-0.1, acts[:, 6].min() - 0.1), max(1.1, acts[:, 6].max() + 0.1))
    _style_ax(ax, xlabel="VLA step [count]", ylabel="gripper command [1]")
    savefig(fig, f"{out_prefix}__actions_gripper", subdir=subdir); plt.show()

def plot_actions_combined(acts, out_name, title, subdir=None):
    x = np.arange(len(acts))
    fig, ax = plt.subplots(figsize=FIG_COMB)
    for j, (lab, c) in enumerate(zip(VLA_LABELS_7, VLA_COLORS_7)):
        ax.plot(x, acts[:, j], label=lab, color=c, lw=1.5)
    ax.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    _style_ax(ax, xlabel="VLA step [count]", ylabel="normalized action [1]")
    ax.legend(fontsize=8, ncol=2)
    savefig(fig, out_name, subdir=subdir); plt.show()

ACTS = load_actions(episode)
if ACTS is not None:
    plot_actions_split(ACTS, TAG)
    plot_actions_combined(ACTS, f"{TAG}__actions_all7", "All 7 actions (representative)")
else:
    print("[WARN] vla_actions missing")

if IS_BATCH:
    arrs = [a for a in (load_actions(e) for e in episodes) if a is not None]
    if arrs:
        L = max(a.shape[0] for a in arrs)
        P = np.full((len(arrs), L, 7), np.nan)
        for i, a in enumerate(arrs): P[i, :a.shape[0]] = a
        mean = np.nanmean(P, axis=0); std = np.nanstd(P, axis=0, ddof=1)
        nobs = np.sum(~np.isnan(P[..., 0]), axis=0)          # ragged horizon: per-step n
        sem = std / np.sqrt(np.maximum(nobs, 1))
        t95 = np.array([ci95_factor(k) if k > 1 else np.nan for k in nobs])
        ci = t95[:, None] * sem
        x = np.arange(L)
        # split + CI
        fig, ax = plt.subplots(figsize=FIG_SQUARE)
        for i, (lab, c) in enumerate(zip(("dx", "dy", "dz"), XYZ_COLORS)):
            ax.plot(x, mean[:, i], label=lab, color=c, lw=1.8)
            ax.fill_between(x, mean[:, i] - ci[:, i], mean[:, i] + ci[:, i], color=c, alpha=CI_ALPHA, label="95% CI" if i == 0 else "_nolegend_")
        ax.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
        _style_ax(ax, xlabel="VLA step [count]", ylabel="normalized action [1]")
        savefig(fig, "batch__actions_xyz", subdir="batch"); plt.show()
        fig, ax = plt.subplots(figsize=FIG_SQUARE)
        for i, (lab, c) in enumerate(zip(("droll", "dpitch", "dyaw"), RPY_COLORS)):
            ax.plot(x, mean[:, 3 + i], label=lab, color=c, lw=1.8)
            ax.fill_between(x, mean[:, 3 + i] - ci[:, 3 + i], mean[:, 3 + i] + ci[:, 3 + i], color=c, alpha=CI_ALPHA, label="95% CI" if i == 0 else "_nolegend_")
        ax.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
        _style_ax(ax, xlabel="VLA step [count]", ylabel="normalized action [1]")
        savefig(fig, "batch__actions_rpy", subdir="batch"); plt.show()
        fig, ax = plt.subplots(figsize=FIG_SQ_SMALL)
        ax.plot(x, mean[:, 6], color=GRIP_COLOR, lw=2.0, label="mean")
        ax.fill_between(x, mean[:, 6] - ci[:, 6], mean[:, 6] + ci[:, 6], color=GRIP_COLOR, alpha=CI_ALPHA, label="95% CI")
        ax.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
        ax.axhline(0.5, color=GRIP_COLOR, lw=0.8, ls=":", alpha=0.7, label="0.5 threshold")
        ax.set_ylim(min(-0.1, np.nanmin(P[:, :, 6]) - 0.1), max(1.1, np.nanmax(P[:, :, 6]) + 0.1))
        _style_ax(ax, xlabel="VLA step [count]", ylabel="gripper command [1]")
        savefig(fig, "batch__actions_gripper", subdir="batch"); plt.show()
        # combined + CI
        plot_actions_combined(mean, "batch__actions_all7",
                              f"All 7 actions — batch mean ± 95% CI (n={len(arrs)})", subdir="batch")
        fig, ax = plt.subplots(figsize=FIG_COMB)
        for j, (lab, c) in enumerate(zip(VLA_LABELS_7, VLA_COLORS_7)):
            ax.plot(x, mean[:, j], label=lab, color=c, lw=1.6)
            ax.fill_between(x, mean[:, j] - ci[:, j], mean[:, j] + ci[:, j], color=c, alpha=CI_ALPHA, label="95% CI" if j == 0 else "_nolegend_")
        ax.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
        _style_ax(ax, xlabel="VLA step [count]", ylabel="normalized action [1]")
        ax.legend(fontsize=8, ncol=2)
        savefig(fig, "batch__actions_all7_ci", subdir="batch"); plt.show()
        np.savez_compressed(TABLE_DIR / "batch_actions_padded.npz",
                            padded=P, mean=mean, std=std, ci95=ci, nobs=nobs)
        print(f"batch actions: {len(arrs)} episodes, horizon {L} (NaN-padded) -> batch_actions_padded.npz")


## 14. VLA inference & wall timing

In [ ]:
def timing_metrics(ep):
    out = {}
    with h5py.File(ep["_path"], "r") as f:
        g = f.get("vla_steps")
        if g is not None and "vla_inference_ms" in g:
            v = np.asarray(g["vla_inference_ms"][()], float)
            out.update(infer_ms_mean=float(v.mean()), infer_ms_max=float(v.max()), _infer=v)
    out["duration_sim_s"] = ep["_dur"]
    A = ep.get("_attrs", {})
    if "completion_time_wall_s" in A: out["duration_wall_s"] = float(A["completion_time_wall_s"])
    return out

TIM = timing_metrics(episode)
print("representative:", {k: round(v, 2) for k, v in TIM.items() if not k.startswith("_")})
if IS_BATCH:
    rows = [timing_metrics(e) for e in episodes]
    for k in ("infer_ms_mean", "infer_ms_max", "duration_sim_s", "duration_wall_s"):
        v = np.array([r.get(k, np.nan) for r in rows], float)
        if np.all(np.isnan(v)): continue
        m, s, ci, n = mean_std_ci(v)
        print(f"batch {k:16s} {m:.2f} ± {s:.2f} (std) ± {ci:.2f} (95% CI)")


## 15. Task success

In [ ]:
print(f"representative success: {get_success(episode)}")
if IS_BATCH:
    v = np.array([1.0 if get_success(e) else 0.0 for e in episodes if get_success(e) is not None], float)
    if len(v):
        rate, n = float(v.mean()), len(v)
        sem = v.std(ddof=1) / np.sqrt(n) if n > 1 else 0.0
        ci = ci95_factor(n) * sem if n > 1 else 0.0
        print(f"batch success: {rate:.0%} ({int(v.sum())}/{n}) ± {ci:.0%} (95% CI)")
        fig, ax = plt.subplots(figsize=(5.2, 4.4))
        ax.bar([0], [100 * rate], yerr=[100 * ci], capsize=8, color="#4477AA",
               edgecolor="white", width=0.5, error_kw=dict(elinewidth=1.4, capthick=1.4))
        ax.set_xticks([0]); ax.set_xticklabels([f"n={n}"])
        ax.set_ylim(0, 105); ax.set_ylabel("success [%]")
        ax.text(0, min(100 * rate + 4, 100), f"{100 * rate:.0f}%", ha="center", fontweight="bold")
        _style_ax(ax, legend=False)
        savefig(fig, "batch__success_ci", subdir="batch"); plt.show()


## 16. Final report — per-episode table + batch aggregate (mean ± std / ± 95% CI)

Exports (to `evaluation_results/tables/`):

* `{TAG}__per_episode_metrics.csv` — one row per run, with units in column headers.
* `{TAG}__metric_units.csv` — explicit metric-to-unit mapping.
* `{TAG}__batch_summary.csv` — one row per metric with a `unit` column, **mean ± std**, **± 95% CI**, and n.
* `{TAG}__rep_raw.json` — representative raw metrics plus the unit map.


In [ ]:
METRIC_UNITS = {
    "stem": "—", "task": "—", "success": "bool",
    "duration_sim_s": "s", "duration_wall_s": "s", "n_ticks": "count",
    "cart_mean_pos_err_m": "m", "cart_max_pos_err_m": "m", "cart_rms_pos_err_m": "m",
    "cart_mean_rot_err_deg": "deg", "cart_max_rot_err_deg": "deg",
    "sm_mean_vel": "m/s", "sm_max_vel": "m/s", "sm_mean_acc": "m/s²", "sm_max_acc": "m/s²",
    "sm_rms_jerk": "m/s³", "sm_joint_max_vel": "rad/s", "sm_joint_rms_jerk": "rad/s³",
    "eff_effort_J": "J", "eff_mean_abs_tau": "N·m", "eff_rms_tau": "N·m",
    "ik_ik_full_pct": "%", "ik_ik_pos_only_pct": "%", "ik_ik_failed_pct": "%",
    "ik_ik_mean_pos_err": "m", "ik_ik_mean_rot_err": "rad", "ik_ik_mean_solve_ms": "ms",
    "tim_infer_ms_mean": "ms", "tim_infer_ms_max": "ms", "tim_duration_sim_s": "s",
    "tim_duration_wall_s": "s", "n_collisions": "count",
}

def final_row(ep):
    A = ep.get("_attrs", {})
    return dict(
        stem=ep["_stem"], task=get_task(ep), success=get_success(ep),
        duration_sim_s=round(ep["_dur"], 3),
        duration_wall_s=round(float(A["completion_time_wall_s"]), 2) if "completion_time_wall_s" in A else None,
        n_ticks=ep["_n_ticks"],
        **{f"cart_{k}": round(v, 5) for k, v in cartesian_metrics(ep).items()},
        **{f"sm_{k}": round(v, 5) for k, v in smoothness_metrics(ep).items() if not k.startswith("_")},
        **{f"eff_{k}": round(v, 4) for k, v in effort_metrics(ep).items()},
        **{f"ik_{k}": round(v, 4) if isinstance(v, float) else v
           for k, v in ik_metrics(ep).items() if not k.startswith("_") and k != "stage_counts"},
        **{f"tim_{k}": round(v, 2) for k, v in timing_metrics(ep).items() if not k.startswith("_")},
        n_collisions=int(np.asarray(ep["collisions"]).astype(bool).sum()) if ep.get("collisions") is not None else None,
    )

rows = [final_row(e) for e in episodes]
per_ep = pd.DataFrame(rows)
units = pd.Series({col: METRIC_UNITS.get(col, "—") for col in per_ep.columns}, name="unit")
if IS_BATCH:
    display(per_ep.rename(columns={col: f"{col} [{units[col]}]" for col in per_ep.columns if units[col] != "—"}))
else:
    final_display = per_ep.T.rename(columns={0: "value"})
    final_display.insert(0, "unit", units.reindex(final_display.index))
    display(final_display)

# Export explicit units in column headers as well as a separate machine-readable unit map.
export_per_ep = per_ep.rename(columns={col: f"{col} [{units[col]}]" for col in per_ep.columns if units[col] != "—"})
export_per_ep.to_csv(TABLE_DIR / f"{TAG}__per_episode_metrics.csv", index=False)
units.rename_axis("metric").reset_index().to_csv(TABLE_DIR / f"{TAG}__metric_units.csv", index=False)
print(f"[table] {TABLE_DIR / (TAG + '__per_episode_metrics.csv')}")
print(f"[table] {TABLE_DIR / (TAG + '__metric_units.csv')}")

if IS_BATCH:
    agg = []
    for col in per_ep.columns:
        if col in ("stem", "task"): continue
        values = pd.to_numeric(per_ep[col], errors="coerce").to_numpy(float)
        if np.all(np.isnan(values)): continue
        mean, std, ci, n = mean_std_ci(values)
        agg.append(dict(metric=col, unit=METRIC_UNITS.get(col, "—"), mean=mean, std=std, ci95=ci, n=int(n),
                        mean_pm_std=f"{mean:.4g} ± {std:.4g}", mean_pm_ci=f"{mean:.4g} ± {ci:.4g}"))
    agg_df = pd.DataFrame(agg)
    display(agg_df)
    agg_df.to_csv(TABLE_DIR / f"{TAG}__batch_summary.csv", index=False)
    print(f"[table] {TABLE_DIR / (TAG + '__batch_summary.csv')}")
    print("\nReporting: tables = mean ± std; figures = mean ± 95% CI (t, df=n-1).")
else:
    print("\n[NOTE] single-episode mode — for reporting set H5_INPUT to the 10-run preset dir.")

with open(TABLE_DIR / f"{TAG}__rep_raw.json", "w") as fp:
    json.dump(dict(representative=dict(cart=CART,
                 smooth={k: v for k, v in SM.items() if not k.startswith("_")},
                 effort=EFF, ik={k: v for k, v in IK.items() if not k.startswith("_")},
                 timing=TIM, success=get_success(episode), task=get_task(episode)),
                 n_files=len(H5_PATHS), sources=SIGNAL_SOURCES, metric_units=METRIC_UNITS),
              fp, indent=2, default=str)
print(f"[table] {TABLE_DIR / (TAG + '__rep_raw.json')}")


---

### Usage

* **Single**: keep `H5_PATH`, leave `H5_INPUT` unset.
* **Batch (10 runs/preset)**: `H5_INPUT = "outputs/runs/<preset>"` (dir or glob) — tables switch to **mean ± std**, figures to **mean ± 95% CI** (`figures/batch/`, `{TAG}__batch_summary.csv`).
* **Video**: exported for every input file — `<stem>.mp4` (60 fps) + `<stem>__history.mp4` (8 fps) under `videos/`.
* Mechanically convertible to `evaluate_episode.py`: wrap sections in `argparse` (`--input`, `--save-figures`).
